# Hierarchical RAG

> Notebook generated from [`examples/rag/06_hierarchical_rag.py`](../../examples/rag/06_hierarchical_rag.py).

| Data | Value |
|------|-------|
| **Dataset** | CUAD (embedded, legal contracts) |
| **API key** | ⚠️  **Requires API key** (`ANTHROPIC_API_KEY` or `OPENAI_API_KEY`) in `.env`. |

**Expected result:** Indexes with child chunks (~100 chars) + parent (~500 chars); shows the parent returned to the LLM after finding the child.

---

## Setup

```bash
uv pip install -e ".[dev,all]"
```

> To use `await main()` directly in cells, this notebook
> installs `nest_asyncio` in the first cell.

---

## Original docstring

```
Hierarchical (Parent-Child) RAG — Long documents with enriched context
=============================================================================
Architecture: SPEC-RAG-005 / prismal.rag.hierarchical

Dataset: CUAD (Contract Understanding Atticus Dataset)
  • 510 commercial contracts with 13,000 legal clause annotations.
  • Reference: https://huggingface.co/datasets/theatticusproject/cuad
  • Why: Legal contracts are long documents with a natural hierarchical structure
    (sections → paragraphs → clauses). Hierarchical RAG indexes small chunks
    (greater precision) but retrieves parent context (greater coherence).
    CUAD is the standard benchmark for contract understanding.

Hierarchical RAG architecture description:
  Two-level indexing:
  - Child chunks  (~100 chars): high granularity for precise matching
  - Parent chunks (~500 chars): rich context for coherent generation

  Process:
  1. Only child chunks are embedded and indexed in ChromaDB
  2. Each child chunk carries metadata: parent_id + parent_content
  3. On search: similarity_search retrieves child chunks
  4. Generation uses the parent content (more context)

  Benefit: High precision of the child + rich parent context.
  Avoids the problem of small chunks with insufficient context for the LLM.

Usage:
    uv run python examples/rag/06_hierarchical_rag.py
```

In [ ]:
# Enable nested event loop to use `await` directly in Jupyter.
import sys
if 'nest_asyncio' not in sys.modules:
    try:
        import nest_asyncio
        nest_asyncio.apply()
    except ImportError:
        %pip install -q nest_asyncio
        import nest_asyncio
        nest_asyncio.apply()

## Setup · imports and dataset

In [ ]:
from __future__ import annotations

import asyncio
import tempfile
from pathlib import Path

from prismal.rag.hierarchical import HierarchicalRAGEngine, HierarchicalSearchResult
from prismal.rag.vector_store import ChromaVectorStore

## Dataset: CUAD contract excerpts

In [ ]:
# Real commercial-contract clauses annotated in CUAD.
CUAD_CONTRACTS = [
    {
        "filename": "software_license_agreement.txt",
        "contract_type": "Software License Agreement",
        "content": (
            "SOFTWARE LICENSE AGREEMENT\n\n"
            "This Software License Agreement ('Agreement') is entered into as of January 1, 2024, "
            "between TechCorp Inc., a Delaware corporation ('Licensor'), and Enterprise Solutions LLC "
            "('Licensee').\n\n"
            "1. GRANT OF LICENSE\n"
            "Subject to the terms and conditions of this Agreement, Licensor hereby grants to "
            "Licensee a non-exclusive, non-transferable, limited license to use the Software solely "
            "for Licensee's internal business purposes. The license does not include the right to "
            "sublicense, modify, adapt, translate, reverse engineer, decompile, disassemble, "
            "or create derivative works based on the Software.\n\n"
            "2. INTELLECTUAL PROPERTY\n"
            "Licensee acknowledges that all intellectual property rights in the Software, "
            "including but not limited to patents, copyrights, trademarks, and trade secrets, "
            "are and shall remain the exclusive property of Licensor. This Agreement does not "
            "transfer any ownership rights in the Software to Licensee.\n\n"
            "3. CONFIDENTIALITY\n"
            "Each party agrees to maintain the confidentiality of the other party's Confidential "
            "Information and not to disclose such information to third parties without prior "
            "written consent. Confidential Information means any information designated as "
            "confidential or that reasonably should be understood to be confidential given the "
            "nature of the information and circumstances of disclosure.\n\n"
            "4. LIMITATION OF LIABILITY\n"
            "IN NO EVENT SHALL LICENSOR BE LIABLE FOR ANY INDIRECT, INCIDENTAL, SPECIAL, "
            "EXEMPLARY, OR CONSEQUENTIAL DAMAGES, INCLUDING BUT NOT LIMITED TO LOSS OF PROFITS, "
            "REVENUE, DATA, OR USE, INCURRED BY LICENSEE OR ANY THIRD PARTY, WHETHER IN AN "
            "ACTION IN CONTRACT OR TORT, EVEN IF LICENSOR HAS BEEN ADVISED OF THE POSSIBILITY "
            "OF SUCH DAMAGES. LICENSOR'S TOTAL CUMULATIVE LIABILITY SHALL NOT EXCEED THE "
            "AMOUNTS PAID BY LICENSEE IN THE TWELVE MONTHS PRECEDING THE CLAIM.\n\n"
            "5. TERM AND TERMINATION\n"
            "This Agreement shall commence on the Effective Date and continue for a period of "
            "one (1) year, unless earlier terminated. Either party may terminate this Agreement "
            "upon thirty (30) days written notice. Licensor may terminate immediately upon "
            "Licensee's material breach that remains uncured for ten (10) business days after "
            "written notice of such breach."
        ),
    },
    {
        "filename": "nda_agreement.txt",
        "contract_type": "Non-Disclosure Agreement",
        "content": (
            "MUTUAL NON-DISCLOSURE AGREEMENT\n\n"
            "This Mutual Non-Disclosure Agreement ('NDA') is entered into by and between "
            "Alpha Innovations Ltd. ('Party A') and Beta Research Corp. ('Party B') "
            "as of March 15, 2024.\n\n"
            "1. DEFINITION OF CONFIDENTIAL INFORMATION\n"
            "For purposes of this Agreement, 'Confidential Information' means any data or "
            "information that is proprietary to the Disclosing Party and not generally known "
            "to the public, whether in tangible or intangible form, whenever and however "
            "disclosed, including, but not limited to: technical data, trade secrets, "
            "research, product plans, products, services, customer lists, markets, "
            "developments, inventions, processes, formulas, technology, designs, drawings, "
            "engineering, hardware configuration information, marketing, finances, or "
            "other business information disclosed by the Disclosing Party.\n\n"
            "2. OBLIGATIONS OF RECEIVING PARTY\n"
            "The Receiving Party agrees to: (a) hold the Confidential Information in strict "
            "confidence; (b) not disclose Confidential Information to any third parties "
            "without prior written approval; (c) use the Confidential Information only for "
            "the purpose of evaluating a potential business relationship between the parties; "
            "(d) take reasonable precautions to prevent unauthorized disclosure or use of "
            "Confidential Information.\n\n"
            "3. TERM\n"
            "This Agreement shall remain in effect for a period of three (3) years from the "
            "date of execution. The obligations of confidentiality shall survive the "
            "termination of this Agreement for a period of five (5) years.\n\n"
            "4. REMEDIES\n"
            "The Receiving Party acknowledges that any breach of this Agreement may cause "
            "irreparable harm to the Disclosing Party for which monetary damages would be "
            "an inadequate remedy. Therefore, the Disclosing Party shall be entitled to seek "
            "equitable relief, including injunction and specific performance, in addition to "
            "all other remedies available at law or in equity."
        ),
    },
    {
        "filename": "service_agreement.txt",
        "contract_type": "Master Service Agreement",
        "content": (
            "MASTER SERVICE AGREEMENT\n\n"
            "This Master Service Agreement ('MSA') is made between CloudServices Inc. "
            "('Provider') and DataDriven Co. ('Client') effective as of June 1, 2024.\n\n"
            "1. SERVICES\n"
            "Provider shall provide Client with cloud computing services, including but not "
            "limited to: Infrastructure as a Service (IaaS), Platform as a Service (PaaS), "
            "and Software as a Service (SaaS) solutions as detailed in applicable Statements "
            "of Work ('SOW'). Each SOW shall specify the scope of services, deliverables, "
            "timelines, and fees for that particular engagement.\n\n"
            "2. SERVICE LEVEL AGREEMENT\n"
            "Provider guarantees a monthly uptime of 99.9% ('SLA'). In the event of SLA "
            "breach, Client shall receive service credits calculated as: (Downtime Minutes / "
            "Total Monthly Minutes) × Monthly Fee × 10. Credits are the sole and exclusive "
            "remedy for SLA breaches. Provider shall provide 48-hour advance notice for "
            "scheduled maintenance windows.\n\n"
            "3. DATA PROTECTION AND SECURITY\n"
            "Provider shall implement and maintain industry-standard security measures, "
            "including SOC 2 Type II compliance, encryption at rest (AES-256) and in transit "
            "(TLS 1.3), role-based access controls, and regular penetration testing. "
            "Provider shall notify Client within 72 hours of discovering any security "
            "incident affecting Client data. All data remains the property of Client.\n\n"
            "4. PAYMENT TERMS\n"
            "Client shall pay all undisputed invoices within net thirty (30) days of receipt. "
            "Late payments shall accrue interest at 1.5% per month. Provider may suspend "
            "services upon sixty (60) days of unpaid invoices following written notice. "
            "All fees are exclusive of applicable taxes."
        ),
    },
]

## Questions about contract clauses (CUAD-style)

In [ ]:
CUAD_QUESTIONS = [
    {
        "id": "CQ1",
        "question": "What is the duration of the confidentiality period in the NDA?",
        "relevant_contract": "nda_agreement.txt",
        "expected_keyword": "five",
        "clause_type": "confidentiality_term",
    },
    {
        "id": "CQ2",
        "question": "What liability limitations does the Licensor apply in the license agreement?",
        "relevant_contract": "software_license_agreement.txt",
        "expected_keyword": "consequential",
        "clause_type": "limitation_of_liability",
    },
    {
        "id": "CQ3",
        "question": "What SLA does CloudServices guarantee and what credits does it offer for breach?",
        "relevant_contract": "service_agreement.txt",
        "expected_keyword": "99.9",
        "clause_type": "service_level_agreement",
    },
    {
        "id": "CQ4",
        "question": "Can the Licensee sublicense or modify the software under the license agreement?",
        "relevant_contract": "software_license_agreement.txt",
        "expected_keyword": "sublicense",
        "clause_type": "grant_of_license",
    },
    {
        "id": "CQ5",
        "question": "What security measures must the cloud services provider implement?",
        "relevant_contract": "service_agreement.txt",
        "expected_keyword": "SOC 2",
        "clause_type": "data_protection",
    },
]


async def setup_hierarchical_rag() -> HierarchicalRAGEngine:
    """Create the Hierarchical RAG engine and index the CUAD contracts.

    Returns:
        HierarchicalRAGEngine with the contracts indexed.
    """
    engine = HierarchicalRAGEngine(
        vector_store=ChromaVectorStore(collection_name="cuad_hierarchical"),
    )

    # Create temporary files and index them
    with tempfile.TemporaryDirectory() as tmpdir:
        tmpdir_path = Path(tmpdir)
        for contract in CUAD_CONTRACTS:
            contract_path = tmpdir_path / contract["filename"]
            contract_path.write_text(contract["content"], encoding="utf-8")
            await asyncio.to_thread(engine.index_document, contract_path)
            print(f"  ✓ Indexed: {contract['filename']} ({contract['contract_type']})")

    return engine


def print_hierarchical_result(question: dict, result: HierarchicalSearchResult) -> None:
    """Print the result with the child/parent difference visible."""
    print(f"\n[{question['id']}] {question['question']}")
    print(f"  Relevant contract: {question['relevant_contract']}")
    print(f"  Clause: {question['clause_type']}")

    print("\n  Retrieved chunks (child → parent):")
    for i, chunk in enumerate(result.chunks[:3]):
        print(f"\n  [{i + 1}] Score: {chunk.relevance_score:.3f}")
        print("  CHILD content (granular, ~100 chars):")
        child_content = chunk.content[:150]
        print(f"    '{child_content}...'")

        # The parent chunk is in the metadata
        parent_content = (
            chunk.metadata.get("parent_content", "") if hasattr(chunk, "metadata") else ""
        )
        if parent_content:
            print("  PARENT content (full context, ~500 chars):")
            print(f"    '{parent_content[:200]}...'")

    # Verify whether the expected keyword is in the results
    all_text = " ".join(c.content.lower() for c in result.chunks[:3])
    expected_kw = question["expected_keyword"].lower()
    found = expected_kw in all_text
    print(
        f"\n  Expected keyword ('{question['expected_keyword']}'): {'✓ found' if found else '✗ not found'}"
    )
    print("─" * 70)


async def main() -> None:
    print("=" * 70)
    print("  Hierarchical RAG — Dataset: CUAD (Commercial Contracts)")
    print("=" * 70)

    # Architecture
    print("\n[Hierarchical indexing architecture]")
    print("  Full document")
    print("  └── Parent chunk (500 chars) ← used to GENERATE the answer")
    print("       └── Child chunk (100 chars) ← indexed in ChromaDB for SEARCH")
    print()
    print("  Advantage: child chunk = search precision")
    print("             parent chunk = rich context for the LLM")

    # Initialization
    print("\n[Indexing CUAD contracts]")
    engine = await setup_hierarchical_rag()
    print("  ✓ Hierarchical RAG engine ready")

    # Run questions
    print(f"\n[Running {len(CUAD_QUESTIONS)} questions about contract clauses]")
    correct = 0

    for question in CUAD_QUESTIONS:
        result = await asyncio.to_thread(engine.search, question["question"], 3)
        print_hierarchical_result(question, result)

        all_text = " ".join(c.content.lower() for c in result.chunks[:3])
        if question["expected_keyword"].lower() in all_text:
            correct += 1

    # Summary
    accuracy = correct / len(CUAD_QUESTIONS)
    print("\n[Summary]")
    print(f"  Keywords found: {correct}/{len(CUAD_QUESTIONS)} ({accuracy:.0%})")

    print("\n[Advantages of hierarchical chunking for legal documents]")
    advantages = [
        ("Precision", "Small chunks match exactly with specific legal terms"),
        (
            "Context",
            "The parent paragraph provides the context needed to interpret the clause",
        ),
        ("No truncation", "No critical context is lost when generating the answer"),
        ("Scalability", "Works for 100+ page contracts without degradation"),
    ]
    for name, desc in advantages:
        print(f"  • {name:15s}: {desc}")

    print("\n[Comparison of chunk sizes]")
    print("  Child chunk : ~100 chars → high search precision")
    print("  Parent chunk: ~500 chars → sufficient context for generation")
    print(
        f"  Document    : {max(len(c['content']) for c in CUAD_CONTRACTS)} chars max → no indexing limit"
    )


if __name__ == "__main__":
    asyncio.run(main())

---

## Run the demo

The original file ends with `asyncio.run(main())`. In Jupyter use:

```python
await main()
```

Thanks to the `nest_asyncio.apply()` in the first cell, the
`async` calls run without needing a separate event loop.

In [ ]:
# Uncomment to run the end-to-end demo (requires API key if applicable).
# await main()